# 22 — Skin myeloid & fibroblast sub-annotation

`10b` split only the T compartment, so the whole CCC pipeline treats **Myeloid (93,435)** and
**Fibroblast (66,760)** as single pooled levels. This notebook splits them and writes the sidecar
`25_delta_gene_axis`/39 consume.

**Flow** — §2–§9 train nothing (§7b trains one small scVI per lineage on the UNK pool, first run only). The per-lineage embeddings (HVG → scVI → UMAP → Leiden)
are already on disk; §2 loads them and only rebuilds if a cache is missing.

| § | what |
|---|---|
| 2 | load the subset + the cached Leiden / UMAP for both lineages |
| 3 | Leiden UMAPs — myeloid and fibroblast |
| 4 | marker dot plots |
| 5 | **the numeric dump**: per cluster, mean expression / detected fraction for every panel gene, signature scores, Wilcoxon top-15, and QC. Copy-pasteable into an LLM. |
| 6 | the annotation dict (hand-filled) → `subtype_fine` |
| 7 | annotated UMAPs |
| 7b–7d | re-cluster the **UNK** pool of both lineages on its own embedding, dump the numbers, relabel (`UNK2FINE`) |
| 8 | `subtype_fine` → `subtype_ccc` collapse, against per-donor coverage |
| 9 | write `skin_myeloid_fibro_subtypes.csv` + provenance |

**Labels are ontogeny/program states, not M1/M2.** M1/M2/TAM are scored as gradients in §5 and never
used as cluster names — they do not partition tissue macrophages. `UNK` is a real answer for a
doublet, pericyte or low-quality cluster.

In [ ]:
# ============================================================================
# §0  Setup
# ============================================================================
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc, json, sys, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from natsort import natsorted

warnings.filterwarnings("ignore")


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir()
sys.path.insert(0, str(NB_DIR / "helpers"))
OUT = NB_DIR / "data" / "atlas_joint"
FIG = NB_DIR / "figures" / "myefib_reannotation"; FIG.mkdir(parents=True, exist_ok=True)
TAB = NB_DIR / "tables"; TAB.mkdir(exist_ok=True)
MODELS = NB_DIR / "models"; MODELS.mkdir(exist_ok=True)

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["savefig.bbox"] = "tight"

LINEAGES = ["Myeloid", "Fibroblast"]
TAGS = {"Myeloid": "mye", "Fibroblast": "fib"}

# ---------------------------------------------------------------- inputs
SRC_H5AD    = OUT / "joint_mrvi_input_skin.h5ad"     # 749,510 x 10,000 — only read if the subset is absent
LABEL_CSV   = OUT / "skin_cell_type_final.csv"       # nb10b
SUBSET_H5AD = OUT / "skin_mye_fib_subset.h5ad"       # 160,195 x 10,000, layers['counts']

# ---------------------------------------------------------------- per-lineage caches
def paths(tag):
    return dict(hvg=OUT / f"skin_{tag}_hvg.json",
                scvi=MODELS / f"skin_{tag}_scvi",
                umap=OUT / f"skin_{tag}_umap.npy",
                leiden=OUT / f"skin_{tag}_leiden.csv",
                deg=TAB / f"skin_{tag}_cluster_markers.csv",
                expr=TAB / f"skin_{tag}_cluster_expression.csv",
                qc=TAB / f"skin_{tag}_cluster_qc.csv",
                cov=TAB / f"skin_{tag}_coverage_by_donor.csv")

# ---------------------------------------------------------------- outputs
SUBTYPE_CSV = OUT / "skin_myeloid_fibro_subtypes.csv"    # what nb38/39 read
PROV_JSON   = OUT / "skin_myeloid_fibro_provenance.json"

# ---------------------------------------------------------------- parameters
N_SKIN, N_MYE, N_FIB = 749_510, 93_435, 66_760
N_CELLS = {"Myeloid": N_MYE, "Fibroblast": N_FIB}
N_HVG, N_LATENT = 2000, 20
HVG_MIN_CELLS_PER_BATCH = 1000          # seurat_v3's loess needs a usable batch
LEIDEN_RES = {"Myeloid": 1.0, "Fibroblast": 0.8}
CCC_MIN_CELLS, CCC_MIN_SAMPLES = 25, 5  # the claim gate, copied from ccc_data
CTCL_DISEASES = ["MF", "SS", "CTCL_other"]

print("caches:")
for lin in LINEAGES:
    p = paths(TAGS[lin])
    print(f"  {lin:11s} " + "  ".join(
        f"{k}={'OK' if v.exists() else 'MISSING'}" for k, v in p.items() if k in
        ("hvg", "scvi", "umap", "leiden")))
print(f"  subset      {'OK' if SUBSET_H5AD.exists() else 'MISSING'}")

In [ ]:
# ============================================================================
# §1  Marker panels, signatures, vocabulary
# ============================================================================
MARKERS = {
 "Myeloid": {
    "pan-myeloid":   ["PTPRC", "LYZ", "AIF1", "HLA-DRA", "CD68", "ITGAX"],
    "LC":            ["CD207", "CD1A", "EPCAM", "CLDN1", "PLEK2"],
    "cDC1":          ["CLEC9A", "XCR1", "WDFY4", "BATF3", "IRF8", "CADM1"],
    "cDC2":          ["CD1C", "CLEC10A", "FCER1A", "CD1B", "IRF4"],
    "DC_LAMP3":      ["LAMP3", "CCR7", "FSCN1", "CCL19", "CCL22", "IDO1", "BIRC3"],
    "pDC":           ["LILRA4", "IL3RA", "TCF4", "GZMB", "JCHAIN"],
    "Mono":          ["S100A8", "S100A9", "S100A12", "FCN1", "VCAN", "CD14", "FCGR3A"],
    "moDC":          ["CD209", "IL1B", "EREG", "THBS1", "CD1C"],
    "Mac_FOLR2":     ["FOLR2", "LYVE1", "SELENOP", "C1QA", "C1QB", "C1QC", "MRC1", "CD163",
                      "F13A1", "STAB1", "MERTK"],
    "Mac_SPP1_TREM2": ["SPP1", "TREM2", "APOC1", "ACP5", "GPNMB", "MMP9", "MMP12", "CHI3L1"],
    "Mac_infl":      ["IL1B", "CXCL1", "CXCL2", "CXCL9", "CXCL10", "CXCL11", "TREM1", "CD300E",
                      "OLR1"],
    "Mac_ISG":       ["ISG15", "IFIT1", "IFI6"],
    "prolif":        ["MKI67", "TOP2A"],
    "!not-myeloid":  ["CD3D", "MS4A1", "KRT14", "COL1A1", "PECAM1", "TPSAB1"],
 },
 "Fibroblast": {
    "pan-fibro":      ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRA"],
    "F_papillary":    ["APCDD1", "ID1", "WIF1", "COL18A1", "PTGDS", "COL6A5", "COL23A1", "HSPB3",
                       "NTN1", "PDPN"],
    "F_reticular":    ["WISP2", "SLPI", "MFAP5", "TSPAN8", "MGP", "CD36", "FMO1", "MYOC", "THY1",
                       "PI16"],
    "F_mesenchymal":  ["ASPN", "POSTN", "GPC3", "SFRP1", "COMP", "COL11A1", "CTHRC1", "LRRC15",
                       "INHBA", "FAP", "TNC"],
    "F_inflammatory": ["CCL19", "APOE", "CXCL2", "CXCL3", "EFEMP1", "CXCL12", "CXCL13", "IL6",
                       "MMP1", "MMP3", "CXCL8", "IL24", "C3", "C7", "CFD"],
    "F_myofibro":     ["ACTA2", "TAGLN", "MYH11", "DES"],
    "F_apCAF":        ["HLA-DRA", "HLA-DQA1", "CD74"],
    "!pericyte/SMC":  ["RGS5", "NOTCH3", "PDGFRB", "KCNJ8"],
    "prolif":         ["MKI67", "TOP2A"],
    "!not-fibro":     ["PTPRC", "CD3D", "LYZ", "KRT14", "PECAM1", "MLANA"],
 },
}

# Graded signatures — reported as numbers in §5, never used as cluster names.
SIGNATURES = {
 "Myeloid": {
    "sig_M1":  ["CXCL9", "CXCL10", "CXCL11", "IL1B", "TNF", "NOS2", "CD86", "IDO1", "SOCS3"],
    "sig_M2":  ["CD163", "MRC1", "MSR1", "STAB1", "F13A1", "SELENOP", "CCL18", "TGFB1", "FOLR2"],
    "sig_TAM": ["SPP1", "TREM2", "GPNMB", "APOC1", "ACP5", "FABP5", "LGMN", "CTSB"],
    "sig_APC": ["HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DQA1", "CD74", "CIITA"],
    "sig_angio": ["VEGFA", "THBS1", "ADM", "SLC2A1"],
 },
 "Fibroblast": {
    "sig_myoCAF": ["POSTN", "COMP", "COL11A1", "CTHRC1", "LRRC15", "FAP", "INHBA", "TNC", "ACTA2"],
    "sig_iCAF":   ["IL6", "CXCL1", "CXCL2", "CXCL8", "CXCL12", "CCL2", "MMP1", "MMP3", "PDGFRA"],
    "sig_apCAF":  ["HLA-DRA", "HLA-DRB1", "HLA-DPA1", "CD74", "CIITA"],
    "sig_ecm":    ["COL1A1", "COL1A2", "COL3A1", "COL5A1", "FN1", "SPARC"],
 },
}

# The only labels §6 may use.
VOCAB = {
 "Myeloid": ["LC", "cDC1", "cDC2", "DC_LAMP3", "pDC", "Mono", "moDC", "Mac_FOLR2",
             "Mac_SPP1_TREM2", "Mac_infl", "Mac_ISG", "Myeloid_prolif", "UNK"],
 "Fibroblast": ["F_papillary", "F_reticular", "F_mesenchymal", "F_inflammatory",
                "F_myofibroblast", "F_apCAF", "F_prolif", "UNK"],
}

for lin in LINEAGES:
    print(f"{lin}: {len(MARKERS[lin])} panels, {len(SIGNATURES[lin])} signatures, "
          f"{len(VOCAB[lin])} labels")

## §2 · Load

`skin_mye_fib_subset.h5ad` (both lineages, 10k genes, raw counts in `layers['counts']`) plus the
cached per-lineage Leiden and UMAP. The Leiden is joined **on `cell_id`**; the UMAP is positional,
so §2b checks it actually agrees with the clustering rather than trusting row order.

The `rebuild` branch (HVG → scVI → neighbors → UMAP → Leiden) only fires when a cache is missing.
It is the heavy path — run it on a GPU kernel or via `bsub`, not on the login node.

In [ ]:
# ============================================================================
# §2a  Load the subset, log1p it, split by lineage
# ============================================================================
if SUBSET_H5AD.exists():
    sub_all = sc.read_h5ad(SUBSET_H5AD)
else:
    print(f"reading {SRC_H5AD.name} (8.9 GB) — first run only ...")
    _a = sc.read_h5ad(SRC_H5AD)
    assert _a.n_obs == N_SKIN and "raw_counts" in _a.layers
    assert _a.obs["cell_id"].is_unique
    lab = pd.read_csv(LABEL_CSV, dtype=str).set_index("cell_id")
    for c in ["cell_type_broad", "cell_type_final"]:
        _a.obs[c] = (lab[c].reindex(_a.obs["cell_id"].astype(str))
                     .astype(str).str.replace("Keartinocyte", "Keratinocyte").to_numpy())
    assert float((_a.obs["cell_type_final"] != "nan").mean()) > 0.999
    sub_all = _a[_a.obs["cell_type_final"].isin(LINEAGES).to_numpy()].copy()
    del _a; gc.collect()
    sub_all.layers["counts"] = sub_all.X.copy()
    del sub_all.layers["raw_counts"]
    for _k in list(sub_all.obsm):
        del sub_all.obsm[_k]
    sub_all.write_h5ad(SUBSET_H5AD, compression="gzip")
    print("wrote", SUBSET_H5AD, sub_all.shape)

_n = sub_all.obs["cell_type_final"].value_counts()
assert int(_n["Myeloid"]) == N_MYE and int(_n["Fibroblast"]) == N_FIB, _n.to_dict()

# X is raw counts in this file -> log1p CP10K for every plot and every number below
if float(sub_all.X[:2000].max()) > 30:
    sc.pp.normalize_total(sub_all, target_sum=1e4)
    sc.pp.log1p(sub_all)
assert float(sub_all.X.max()) < 20, "X does not look log1p-normalized"

A = {}
for lin in LINEAGES:
    a = sub_all[sub_all.obs["cell_type_final"] == lin].copy()
    for c in a.obs.columns:
        if str(a.obs[c].dtype) == "category":
            a.obs[c] = a.obs[c].cat.remove_unused_categories()
    assert a.n_obs == N_CELLS[lin], (a.n_obs, N_CELLS[lin])
    A[lin] = a
del sub_all; gc.collect()

for lin, a in A.items():
    print(f"{lin:11s} {a.n_obs:>7,} cells | {a.obs['donor'].nunique()} donors | "
          f"{a.obs['sample_id'].nunique()} samples | {a.obs['study'].nunique()} studies")

In [ ]:
# ============================================================================
# §2b  Attach the cached Leiden + UMAP   (rebuild only if missing — HEAVY, GPU)
# ============================================================================
def rebuild(a, lin):
    """HVG -> scVI -> neighbors -> UMAP -> Leiden, and freeze all of it. GPU."""
    import scvi, torch
    tag, p = TAGS[lin], paths(TAGS[lin])
    scvi.settings.seed = SEED
    print(f"[{lin}] rebuilding — cuda={torch.cuda.is_available()}")

    if p["hvg"].exists():
        hvg = json.loads(p["hvg"].read_text())["genes"]
    else:
        n_by = a.obs["study"].value_counts()
        keep_studies = n_by[n_by >= HVG_MIN_CELLS_PER_BATCH].index.tolist()
        _fit = a[a.obs["study"].isin(keep_studies)].copy()
        sc.pp.highly_variable_genes(_fit, flavor="seurat_v3", n_top_genes=N_HVG,
                                    batch_key="study", layer="counts")
        hvg = _fit.var_names[_fit.var["highly_variable"].to_numpy()].tolist()
        del _fit; gc.collect()
        p["hvg"].write_text(json.dumps(
            {"lineage": lin, "n_top_genes": N_HVG, "flavor": "seurat_v3", "batch_key": "study",
             "hvg_studies": keep_studies, "seed": SEED, "genes": hvg}, indent=1))

    ahv = a[:, hvg].copy()
    scvi.model.SCVI.setup_anndata(ahv, layer="counts", batch_key="study",
                                  categorical_covariate_keys=["donor"])
    if p["scvi"].exists():
        model = scvi.model.SCVI.load(str(p["scvi"]), adata=ahv)
    else:
        model = scvi.model.SCVI(ahv, n_latent=N_LATENT, n_layers=2, gene_likelihood="nb")
        model.train(early_stopping=True, early_stopping_patience=15,
                    plan_kwargs={"lr": 1e-3}, check_val_every_n_epoch=1)
        model.save(str(p["scvi"]), overwrite=True)
    a.obsm["X_scVI"] = model.get_latent_representation()
    del ahv, model; gc.collect()

    sc.pp.neighbors(a, use_rep="X_scVI", random_state=SEED)
    sc.tl.umap(a, random_state=SEED)
    np.save(p["umap"], a.obsm["X_umap"])
    sc.tl.leiden(a, resolution=LEIDEN_RES[lin], random_state=SEED, key_added=f"leiden_{tag}",
                 flavor="igraph", n_iterations=2, directed=False)
    pd.DataFrame({"cell_id": a.obs["cell_id"].astype(str).to_numpy(),
                  f"leiden_{tag}": a.obs[f"leiden_{tag}"].astype(str).to_numpy()}
                 ).to_csv(p["leiden"], index=False)
    print(f"[{lin}] rebuilt: {a.obs[f'leiden_{tag}'].nunique()} clusters")


KEY = {}
for lin in LINEAGES:
    tag, p = TAGS[lin], paths(TAGS[lin])
    a, KEY[lin] = A[lin], f"leiden_{tag}"

    if not (p["umap"].exists() and p["leiden"].exists()):
        rebuild(a, lin)
    else:
        # Leiden: joined on cell_id, so row order cannot corrupt it.
        s = pd.read_csv(p["leiden"], dtype=str).set_index("cell_id")[f"leiden_{tag}"]
        ids = a.obs["cell_id"].astype(str)
        assert len(s) == a.n_obs and ids.isin(s.index).all(), (
            f"{p['leiden'].name} does not cover {lin} ({len(s):,} vs {a.n_obs:,})")
        vals = s.reindex(ids).to_numpy()
        a.obs[KEY[lin]] = pd.Categorical(vals, categories=natsorted(pd.unique(vals)))

        # UMAP: stored positionally. Verify it belongs to THIS clustering rather than trusting
        # the row order — cells of one cluster must sit far tighter than the cloud as a whole.
        u = np.load(p["umap"])
        assert u.shape[0] == a.n_obs, (u.shape, a.n_obs)
        a.obsm["X_umap"] = u
        lab = a.obs[KEY[lin]].astype(str).to_numpy()
        big = pd.Series(lab).value_counts().idxmax()
        m = lab == big
        spread = np.median(np.linalg.norm(u[m] - u[m].mean(0), axis=1))
        overall = np.median(np.linalg.norm(u - u.mean(0), axis=1))
        assert spread < 0.6 * overall, (
            f"{lin}: UMAP does not match the Leiden (cluster {big} spread {spread:.2f} vs "
            f"{overall:.2f} overall) — the .npy row order is stale, delete it and rebuild")
        print(f"{lin:11s} loaded: {a.obs[KEY[lin]].nunique()} clusters, UMAP OK "
              f"(spread {spread:.2f} vs {overall:.2f})")

for lin in LINEAGES:
    print(f"\n{lin} cluster sizes:")
    print(A[lin].obs[KEY[lin]].value_counts()
          .reindex(natsorted(A[lin].obs[KEY[lin]].cat.categories)).to_string())

## §3 · Leiden UMAPs

In [ ]:
# ============================================================================
# §3  UMAPs — myeloid and fibroblast
# ============================================================================
for lin in LINEAGES:
    a = A[lin]
    fig = sc.pl.umap(a, color=[KEY[lin], "study", "disease"], ncols=3, frameon=False, size=3,
                     wspace=0.25, legend_loc="on data", legend_fontsize=7, show=False,
                     return_fig=True)
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    fig.suptitle(f"{lin}: {a.n_obs:,} cells, {a.obs[KEY[lin]].nunique()} clusters "
                 f"(res={LEIDEN_RES[lin]})", y=1.02, fontsize=10)
    fig.savefig(FIG / f"{TAGS[lin]}_umap_leiden.png", dpi=200)
    plt.show(); plt.close(fig)

## §4 · Marker dot plots

In [ ]:
# ============================================================================
# §4  Dot plots (the panels of §1, standard-scaled per gene)
# ============================================================================
PANEL = {}
for lin in LINEAGES:
    present = set(A[lin].var_names)
    PANEL[lin] = {k: [g for g in v if g in present] for k, v in MARKERS[lin].items()}
    dropped = {k: [g for g in v if g not in present] for k, v in MARKERS[lin].items()}
    PANEL[lin] = {k: v for k, v in PANEL[lin].items() if v}
    for k, v in dropped.items():
        if v:
            print(f"{lin}: dropped from {k}: {v}")

for lin in LINEAGES:
    dp = sc.pl.dotplot(A[lin], PANEL[lin], groupby=KEY[lin], standard_scale="var",
                       dendrogram=True, figsize=(16, 5), return_fig=True,
                       title=f"{lin} — marker panels by cluster")
    dp.savefig(FIG / f"{TAGS[lin]}_dotplot_markers.png", dpi=200)
    dp.show()

## §5 · The numbers — copy-paste this into an LLM

For every cluster: size / donor spread / QC, the five graded signature scores, **mean log1p
expression and detected fraction (`mean/frac`) for every panel gene**, the li2024 label it overlaps
most, and the Wilcoxon top-15. Nothing is thresholded away — a marker that is absent shows as
`0.00/0.00`, which is evidence too.

The same numbers go to `tables/skin_{mye,fib}_cluster_expression.csv` if you'd rather work from the
file.

In [ ]:
# ============================================================================
# §5a  Per-cluster expression / signature / QC tables
# ============================================================================
EXPR, QC, TOPN = {}, {}, {}

for lin in LINEAGES:
    a, key, p = A[lin], KEY[lin], paths(TAGS[lin])
    genes = [g for v in PANEL[lin].values() for g in v]
    genes = list(dict.fromkeys(genes))

    # mean log1p and detected fraction, per cluster x panel gene
    X = a[:, genes].X
    X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
    grp = a.obs[key].astype(str).to_numpy()
    mean = pd.DataFrame(X, columns=genes).groupby(grp).mean()
    frac = pd.DataFrame((X > 0).astype(np.float32), columns=genes).groupby(grp).mean()
    order = natsorted(mean.index)
    EXPR[lin] = (mean.reindex(order).round(2), frac.reindex(order).round(2))
    del X; gc.collect()

    # graded signatures
    for name, gl in SIGNATURES[lin].items():
        gl = [g for g in gl if g in a.var_names]
        sc.tl.score_genes(a, gl, score_name=name, random_state=SEED)
    sigs = a.obs.groupby(key, observed=True)[list(SIGNATURES[lin])].mean().reindex(order).round(2)

    # QC + the li2024 external anchor
    g = a.obs.groupby(key, observed=True)
    qc = pd.DataFrame(index=pd.Index(order, name=key))
    qc["n"] = g.size()
    qc["n_donors"] = g["donor"].nunique()
    qc["n_studies"] = g["study"].nunique()
    qc["top_study"] = g["study"].apply(lambda s: s.value_counts().idxmax())
    qc["top_study_frac"] = g["study"].apply(lambda s: round(float(s.value_counts(normalize=True).iloc[0]), 2))
    qc["med_genes"] = g["n_genes"].median().astype(int)
    qc["med_mito"] = g["pct_mito"].median().round(1)
    qc["med_doublet"] = g["doublet_score"].median().round(2)
    known = a.obs[~a.obs["cell_type"].astype(str).isin(["Unknown", "nan", ""])]
    if len(known):
        ct = pd.crosstab(known[key].astype(str), known["cell_type"].astype(str))
        fr = ct.div(ct.sum(1).clip(lower=1), axis=0)
        qc["li2024_top"] = fr.idxmax(1).reindex(order)
        qc["li2024_frac"] = fr.max(1).round(2).reindex(order)
    QC[lin] = qc.join(sigs)

    # Wilcoxon top-15 (cached — it is the slow step here)
    if p["deg"].exists():
        deg = pd.read_csv(p["deg"], dtype={"group": str})
    else:
        sc.tl.rank_genes_groups(a, key, method="wilcoxon", pts=True)
        deg = sc.get.rank_genes_groups_df(a, group=None)
        deg = deg[(deg["pvals_adj"] < 0.01) & (deg["logfoldchanges"] > 0.5)]
        deg = deg.sort_values(["group", "scores"], ascending=[True, False])
        deg.to_csv(p["deg"], index=False)
    TOPN[lin] = (deg.groupby("group", observed=True).head(15)
                 .groupby("group", observed=True)["names"].apply(", ".join).reindex(order))

    long = (EXPR[lin][0].stack().rename("mean_log1p").to_frame()
            .join(EXPR[lin][1].stack().rename("frac_detected")))
    long.index.names = [key, "gene"]
    long.to_csv(p["expr"])
    QC[lin].to_csv(p["qc"])
    print(f"{lin}: {len(order)} clusters, {len(genes)} panel genes -> "
          f"{p['expr'].name}, {p['qc'].name}")

In [ ]:
# ============================================================================
# §5b  The paste block
# ============================================================================
for lin in LINEAGES:
    a, key = A[lin], KEY[lin]
    mean, frac = EXPR[lin]
    qc = QC[lin]
    print("=" * 100)
    print(f"{lin.upper()} — {a.n_obs:,} skin cells, {len(mean)} Leiden clusters "
          f"(res={LEIDEN_RES[lin]}), values are `mean log1p CP10K / fraction detected`")
    print(f"allowed labels: {VOCAB[lin]}")
    print("=" * 100)
    for c in mean.index:
        r = qc.loc[c]
        head = (f"\n--- cluster {c}  n={int(r['n']):,}  donors={int(r['n_donors'])}  "
                f"studies={int(r['n_studies'])}  top_study={r['top_study']}({r['top_study_frac']})  "
                f"med_genes={int(r['med_genes'])}  med_mito={r['med_mito']}  "
                f"doublet={r['med_doublet']}")
        if "li2024_top" in qc.columns:
            head += f"  li2024={r['li2024_top']}({r['li2024_frac']})"
        print(head)
        print("  signatures: " + "  ".join(f"{s}={r[s]}" for s in SIGNATURES[lin]))
        for panel, genes in PANEL[lin].items():
            vals = "  ".join(f"{g} {mean.loc[c, g]:.2f}/{frac.loc[c, g]:.2f}" for g in genes)
            print(f"  {panel:15s} {vals}")
        print(f"  wilcoxon top15 : {TOPN[lin].get(c, '')}")
    print()

## §6 · The annotation dict

Fill both dicts from §5. Rules that keep this honest:

- every cluster ID must appear, and every label must come from `VOCAB` — `UNK` for doublet /
  pericyte / low-quality clusters (`RGS5`/`NOTCH3` on a fibroblast cluster, `CD3D`/`KRT14` on a
  myeloid one, `med_genes` < 500 or `med_mito` > 15, `top_study_frac` > 0.9 with no distinctive
  Wilcoxon genes);
- `CLUSTER_SIZES_*` locks the map to this exact clustering. Leave it empty on the first pass — the
  cell prints the sizes to paste. If the Leiden is ever recomputed the sizes change and the assert
  fires instead of silently relabelling different cells.

In [ ]:
# ============================================================================
# §6  cluster -> subtype_fine   (HAND-FILLED from §5)
# ============================================================================
CLUSTER2FINE = {
    "Myeloid": {"0": "LC", "1": "cDC2", "2": "DC_LAMP3", "3": "moDC", "4": "Mac_infl", "5": "cDC2", "6": "DC_LAMP3", "7": "Mono", "8": "Mono", "9": "cDC2", "10": "DC_LAMP3", "11": "Mac_infl", "12": "Myeloid_prolif", "13": "Mac_SPP1_TREM2", "14": "UNK", "15": "Mac_FOLR2", "16": "UNK", "17": "Mac_ISG", "18": "pDC", "19": "cDC1", "20": "UNK", "21": "cDC2", "22": "DC_LAMP3"},
    "Fibroblast": {"0": "F_inflammatory", "1": "F_inflammatory", "2": "F_reticular", "3": "F_inflammatory", "4": "F_apCAF", "5": "UNK", "6": "F_myofibroblast", "7": "F_reticular", "8": "F_apCAF", "9": "UNK", "10": "F_papillary", "11": "F_mesenchymal", "12": "F_papillary", "13": "UNK", "14": "UNK", "15": "F_reticular", "16": "UNK", "17": "UNK"},
}
CLUSTER_SIZES = {
    "Myeloid": {'0': 2861, '1': 3166, '2': 8072, '3': 1313, '4': 5715, '5': 7136, '6': 1635, '7': 20, '8': 5714, '9': 8734, '10': 6458, '11': 1798, '12': 1060, '13': 3835, '14': 432, '15': 12685, '16': 4305, '17': 4086, '18': 1113, '19': 8001, '20': 73, '21': 2320, '22': 2903},
    "Fibroblast": {'0': 3764, '1': 5221, '2': 9482, '3': 4374, '4': 7777, '5': 959, '6': 2730, '7': 12289, '8': 39, '9': 2257, '10': 5720, '11': 3416, '12': 1310, '13': 1801, '14': 4919, '15': 262, '16': 433, '17': 7},
}

MAPPED = True
for lin in LINEAGES:
    a, key = A[lin], KEY[lin]
    cats = [str(c) for c in a.obs[key].cat.categories]
    now = {c: int((a.obs[key].astype(str) == c).sum()) for c in cats}
    m = CLUSTER2FINE[lin]

    if (not m) or any(v == "" for v in m.values()):
        print(f"--- {lin}: UNFILLED. Paste into CLUSTER2FINE[{lin!r}]: ---")
        print("    {" + ", ".join(f'"{c}": ""' for c in cats) + "}")
        print(f"--- and into CLUSTER_SIZES[{lin!r}]: ---\n    {now!r}\n")
        MAPPED = False
        continue

    assert set(m) == set(cats), (
        f"{lin}: map keys != clusters\n  missing: {natsorted(set(cats) - set(m))}\n"
        f"  extra  : {natsorted(set(m) - set(cats))}")
    bad = set(m.values()) - set(VOCAB[lin])
    assert not bad, f"{lin}: labels outside the vocabulary: {sorted(bad)}"
    if CLUSTER_SIZES[lin]:
        assert now == {str(k): int(v) for k, v in CLUSTER_SIZES[lin].items()}, (
            f"{lin}: the Leiden changed since this map was authored — cluster IDs no longer mean "
            f"what the map says.\n  authored: {CLUSTER_SIZES[lin]}\n  now: {now}")
    else:
        print(f"!! {lin}: CLUSTER_SIZES is empty — paste {now!r} to lock the map")

    a.obs["subtype_fine"] = pd.Categorical(a.obs[key].astype(str).map(m))
    print(f"{lin}:")
    print(a.obs["subtype_fine"].value_counts().to_string())

if not MAPPED:
    print("\nFill the dicts above, then re-run from here — §7 onwards needs subtype_fine.")

## §7 · Annotated UMAPs

In [ ]:
# ============================================================================
# §7  Annotated UMAPs + the labelled dot plot
# ============================================================================
assert MAPPED, "fill CLUSTER2FINE in §6 first"

for lin in LINEAGES:
    a = A[lin]
    fig = sc.pl.umap(a, color="subtype_fine", frameon=False, size=3, legend_loc="on data",
                     legend_fontsize=7, show=False, return_fig=True,
                     title=f"{lin} — subtype_fine ({a.obs['subtype_fine'].nunique()} states)")
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    fig.savefig(FIG / f"{TAGS[lin]}_umap_subtype_fine.png", dpi=200)
    plt.show(); plt.close(fig)

    dp = sc.pl.dotplot(a, PANEL[lin], groupby="subtype_fine", standard_scale="var",
                       dendrogram=True, figsize=(16, 4.5), return_fig=True,
                       title=f"{lin} — panels by subtype_fine")
    dp.savefig(FIG / f"{TAGS[lin]}_dotplot_subtype_fine.png", dpi=200)
    dp.show()

## §7b · Re-cluster the UNK

`UNK` in §6 is a cluster-level verdict, not a cell-level one: at res≈1.0 a doublet-looking cluster
still carries real LC / macrophage / fibroblast cells, and the genuinely off-lineage cells
(pericyte, keratinocyte, T, mast) sit mixed in with them. This section takes the UNK cells of
**both lineages**, embeds each pool on its own (UNK-only HVG → scVI → UMAP → Leiden, so the
variation that was previously swamped by the dominant states can separate), dumps the same numeric
block as §5 plus an off-lineage marker panel, and lets §7d relabel the sub-clusters.

- the UNK pool is defined from `CLUSTER2FINE` (the cluster map), not from `subtype_fine`, so this
  section is idempotent — re-running it after §7d does not shrink the pool;
- §7d rebuilds `subtype_fine` from `CLUSTER2FINE` first and then applies the refinement, so the
  labels never stack;
- labels still come from `VOCAB[lin]` only. A sub-cluster that really is pericyte / keratinocyte /
  T / doublet stays `UNK` — the point is to rescue the cells that are not.

**First run trains** (one small scVI per lineage, ~5k / ~10k cells) — run it on the GPU kernel.
After that everything is read from `skin_{mye,fib}_unk_embedding.csv`.

In [ ]:
# ============================================================================
# §7b  Re-cluster the UNK cells of both lineages   (first run TRAINS — GPU)
# ============================================================================
assert MAPPED, "fill CLUSTER2FINE in §6 first"

UNK_RES = {"Myeloid": 0.6, "Fibroblast": 0.6}
UNK_N_HVG, UNK_N_LATENT = 1500, 10
UNK_MIN_CELLS = 200          # below this the pool is too small to re-cluster honestly


def unk_paths(tag):
    return dict(hvg=OUT / f"skin_{tag}_unk_hvg.json",
                scvi=MODELS / f"skin_{tag}_unk_scvi",
                emb=OUT / f"skin_{tag}_unk_embedding.csv",   # cell_id, umap1/2, leiden — one file,
                deg=TAB / f"skin_{tag}_unk_cluster_markers.csv",   # joined on cell_id so the row
                expr=TAB / f"skin_{tag}_unk_cluster_expression.csv",  # order cannot corrupt it
                qc=TAB / f"skin_{tag}_unk_cluster_qc.csv")


def embed_unk(u, lin):
    """UNK-only HVG -> scVI -> UMAP -> Leiden, frozen into one cell_id-indexed csv. GPU."""
    import scvi, torch
    tag, p = TAGS[lin], unk_paths(TAGS[lin])
    scvi.settings.seed = SEED
    print(f"[{lin} UNK] embedding {u.n_obs:,} cells — cuda={torch.cuda.is_available()}")

    if p["hvg"].exists():
        hvg = json.loads(p["hvg"].read_text())["genes"]
    else:
        _fit = u.copy()
        sc.pp.highly_variable_genes(_fit, flavor="seurat_v3", n_top_genes=UNK_N_HVG,
                                    layer="counts")
        hvg = _fit.var_names[_fit.var["highly_variable"].to_numpy()].tolist()
        del _fit; gc.collect()
        p["hvg"].write_text(json.dumps(
            {"lineage": lin, "subset": "UNK", "n_top_genes": UNK_N_HVG, "flavor": "seurat_v3",
             "batch_key": None, "seed": SEED, "genes": hvg}, indent=1))

    uhv = u[:, hvg].copy()
    scvi.model.SCVI.setup_anndata(uhv, layer="counts", batch_key="study")
    if p["scvi"].exists():
        model = scvi.model.SCVI.load(str(p["scvi"]), adata=uhv)
    else:
        model = scvi.model.SCVI(uhv, n_latent=UNK_N_LATENT, n_layers=2, gene_likelihood="nb")
        model.train(early_stopping=True, early_stopping_patience=15,
                    plan_kwargs={"lr": 1e-3}, check_val_every_n_epoch=1)
        model.save(str(p["scvi"]), overwrite=True)
    u.obsm["X_scVI_unk"] = model.get_latent_representation()
    del uhv, model; gc.collect()

    sc.pp.neighbors(u, use_rep="X_scVI_unk", random_state=SEED, key_added="unk")
    sc.tl.umap(u, random_state=SEED, neighbors_key="unk")
    sc.tl.leiden(u, resolution=UNK_RES[lin], random_state=SEED, key_added=f"leiden_{tag}_unk",
                 flavor="igraph", n_iterations=2, directed=False, neighbors_key="unk")
    pd.DataFrame({"cell_id": u.obs["cell_id"].astype(str).to_numpy(),
                  "umap1": u.obsm["X_umap"][:, 0].astype(np.float32),
                  "umap2": u.obsm["X_umap"][:, 1].astype(np.float32),
                  f"leiden_{tag}_unk": u.obs[f"leiden_{tag}_unk"].astype(str).to_numpy()}
                 ).to_csv(p["emb"], index=False)
    print(f"[{lin} UNK] {u.obs[f'leiden_{tag}_unk'].nunique()} sub-clusters -> {p['emb'].name}")


U, UKEY = {}, {}
for lin in LINEAGES:
    a, tag, p = A[lin], TAGS[lin], unk_paths(TAGS[lin])
    UKEY[lin] = f"leiden_{tag}_unk"

    # from the cluster map, not from subtype_fine -> unchanged by §7d
    unk_clusters = [c for c, v in CLUSTER2FINE[lin].items() if v == "UNK"]
    m = a.obs[KEY[lin]].astype(str).isin(unk_clusters).to_numpy()
    if int(m.sum()) < UNK_MIN_CELLS:
        print(f"{lin}: {int(m.sum()):,} UNK cells (< {UNK_MIN_CELLS}) — not re-clustered")
        continue

    u = a[m].copy()
    u.obs["parent_cluster"] = a.obs[KEY[lin]].astype(str).to_numpy()[m]
    for c in u.obs.columns:
        if str(u.obs[c].dtype) == "category":
            u.obs[c] = u.obs[c].cat.remove_unused_categories()

    if p["emb"].exists():
        e = pd.read_csv(p["emb"], dtype={"cell_id": str}).set_index("cell_id")
        ids = u.obs["cell_id"].astype(str)
        assert len(e) == u.n_obs and ids.isin(e.index).all(), (
            f"{p['emb'].name} covers {len(e):,} cells but the UNK pool is {u.n_obs:,} — §6 changed "
            f"which clusters are UNK; delete the cache (and {p['scvi'].name}) and re-run")
        e = e.reindex(ids)
        u.obsm["X_umap"] = e[["umap1", "umap2"]].to_numpy()
        vals = e[UKEY[lin]].astype(str).to_numpy()
        u.obs[UKEY[lin]] = pd.Categorical(vals, categories=natsorted(pd.unique(vals)))
    else:
        embed_unk(u, lin)

    U[lin] = u
    print(f"{lin:11s} UNK {u.n_obs:>6,} cells ({u.n_obs / a.n_obs:.1%} of the lineage) from "
          f"parent clusters {natsorted(unk_clusters)} -> {u.obs[UKEY[lin]].nunique()} sub-clusters")

for lin, u in U.items():
    fig = sc.pl.umap(u, color=[UKEY[lin], "parent_cluster", "study", "disease"], ncols=4,
                     frameon=False, size=6, wspace=0.25, legend_loc="on data", legend_fontsize=7,
                     show=False, return_fig=True)
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    fig.suptitle(f"{lin} UNK: {u.n_obs:,} cells, {u.obs[UKEY[lin]].nunique()} sub-clusters "
                 f"(res={UNK_RES[lin]})", y=1.03, fontsize=10)
    fig.savefig(FIG / f"{TAGS[lin]}_unk_umap_leiden.png", dpi=200)
    plt.show(); plt.close(fig)

In [ ]:
# ============================================================================
# §7c  The numbers for the UNK sub-clusters — copy-paste this into an LLM
# ============================================================================
# The lineage panels of §1 plus the identities an UNK cluster is usually made of.
OFF_MARKERS = {
 "T/NK":         ["CD3D", "CD3E", "TRBC2", "IL7R", "NKG7", "GNLY"],
 "B/plasma":     ["MS4A1", "CD79A", "MZB1", "JCHAIN"],
 "Mast":         ["TPSAB1", "TPSB2", "CPA3", "KIT"],
 "Keratinocyte": ["KRT14", "KRT5", "KRT1", "KRT10", "DMKN", "KRTDAP"],
 "Endothelial":  ["PECAM1", "VWF", "CDH5", "PROX1", "CCL21"],
 "Pericyte/SMC": ["RGS5", "NOTCH3", "PDGFRB", "KCNJ8", "MYH11"],
 "Melanocyte":   ["MLANA", "PMEL", "TYRP1", "DCT"],
 "Neural/Schwann": ["PLP1", "S100B", "NRXN1"],
 "Erythroid":    ["HBB", "HBA1"],
 "myeloid-core": ["PTPRC", "LYZ", "AIF1", "HLA-DRA", "CD68", "C1QA", "ITGAX"],
 "fibro-core":   ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA"],
 "stress":       ["HSPA1A", "HSPA1B", "FOS", "JUN", "EGR1", "MT-CO1"],
}

UNK_PANEL, UEXPR, UQC, UTOPN = {}, {}, {}, {}

for lin, u in U.items():
    key, p = UKEY[lin], unk_paths(TAGS[lin])
    present = set(u.var_names)
    panel = {**PANEL[lin], **{k: [g for g in v if g in present] for k, v in OFF_MARKERS.items()}}
    UNK_PANEL[lin] = {k: v for k, v in panel.items() if v}
    genes = list(dict.fromkeys(g for v in UNK_PANEL[lin].values() for g in v))

    X = u[:, genes].X
    X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
    grp = u.obs[key].astype(str).to_numpy()
    mean = pd.DataFrame(X, columns=genes).groupby(grp).mean()
    frac = pd.DataFrame((X > 0).astype(np.float32), columns=genes).groupby(grp).mean()
    order = natsorted(mean.index)
    UEXPR[lin] = (mean.reindex(order).round(2), frac.reindex(order).round(2))
    del X; gc.collect()

    for name, gl in SIGNATURES[lin].items():
        gl = [g for g in gl if g in u.var_names]
        sc.tl.score_genes(u, gl, score_name=name, random_state=SEED)
    sigs = u.obs.groupby(key, observed=True)[list(SIGNATURES[lin])].mean().reindex(order).round(2)

    g = u.obs.groupby(key, observed=True)
    qc = pd.DataFrame(index=pd.Index(order, name=key))
    qc["n"] = g.size()
    qc["n_donors"] = g["donor"].nunique()
    qc["n_studies"] = g["study"].nunique()
    qc["top_study"] = g["study"].apply(lambda s: s.value_counts().idxmax())
    qc["top_study_frac"] = g["study"].apply(lambda s: round(float(s.value_counts(normalize=True).iloc[0]), 2))
    qc["med_genes"] = g["n_genes"].median().astype(int)
    qc["med_mito"] = g["pct_mito"].median().round(1)
    qc["med_doublet"] = g["doublet_score"].median().round(2)
    qc["parent"] = g["parent_cluster"].apply(
        lambda s: "+".join(f"{k}:{v}" for k, v in s.value_counts(normalize=True).round(2)
                           .head(3).items()))
    known = u.obs[~u.obs["cell_type"].astype(str).isin(["Unknown", "nan", ""])]
    if len(known):
        ct = pd.crosstab(known[key].astype(str), known["cell_type"].astype(str))
        fr = ct.div(ct.sum(1).clip(lower=1), axis=0)
        qc["li2024_top"] = fr.idxmax(1).reindex(order)
        qc["li2024_frac"] = fr.max(1).round(2).reindex(order)
    UQC[lin] = qc.join(sigs)

    if p["deg"].exists():
        deg = pd.read_csv(p["deg"], dtype={"group": str})
    else:
        sc.tl.rank_genes_groups(u, key, method="wilcoxon", pts=True)
        deg = sc.get.rank_genes_groups_df(u, group=None)
        deg = deg[(deg["pvals_adj"] < 0.01) & (deg["logfoldchanges"] > 0.5)]
        deg = deg.sort_values(["group", "scores"], ascending=[True, False])
        deg.to_csv(p["deg"], index=False)
    UTOPN[lin] = (deg.groupby("group", observed=True).head(15)
                  .groupby("group", observed=True)["names"].apply(", ".join).reindex(order))

    long = (UEXPR[lin][0].stack().rename("mean_log1p").to_frame()
            .join(UEXPR[lin][1].stack().rename("frac_detected")))
    long.index.names = [key, "gene"]
    long.to_csv(p["expr"])
    UQC[lin].to_csv(p["qc"])
    print(f"{lin} UNK: {len(order)} sub-clusters, {len(genes)} panel genes -> "
          f"{p['expr'].name}, {p['qc'].name}")

for lin, u in U.items():
    key = UKEY[lin]
    mean, frac = UEXPR[lin]
    qc = UQC[lin]
    print("=" * 100)
    print(f"{lin.upper()} UNK POOL — {u.n_obs:,} cells that §6 could not call, "
          f"{len(mean)} sub-clusters (res={UNK_RES[lin]}), values are "
          f"`mean log1p CP10K / fraction detected`")
    print(f"allowed labels: {VOCAB[lin]}   (leave a sub-cluster UNK if it is off-lineage, "
          f"a doublet, or low-quality)")
    print("=" * 100)
    for c in mean.index:
        r = qc.loc[c]
        head = (f"\n--- UNK sub-cluster {c}  n={int(r['n']):,}  donors={int(r['n_donors'])}  "
                f"studies={int(r['n_studies'])}  top_study={r['top_study']}({r['top_study_frac']})  "
                f"med_genes={int(r['med_genes'])}  med_mito={r['med_mito']}  "
                f"doublet={r['med_doublet']}  parent={r['parent']}")
        if "li2024_top" in qc.columns:
            head += f"  li2024={r['li2024_top']}({r['li2024_frac']})"
        print(head)
        print("  signatures: " + "  ".join(f"{s}={r[s]}" for s in SIGNATURES[lin]))
        for panel, genes in UNK_PANEL[lin].items():
            vals = "  ".join(f"{g} {mean.loc[c, g]:.2f}/{frac.loc[c, g]:.2f}" for g in genes)
            print(f"  {panel:15s} {vals}")
        print(f"  wilcoxon top15 : {UTOPN[lin].get(c, '')}")
    print()

for lin, u in U.items():
    dp = sc.pl.dotplot(u, UNK_PANEL[lin], groupby=UKEY[lin], standard_scale="var",
                       dendrogram=True, figsize=(20, 4.5), return_fig=True,
                       title=f"{lin} UNK pool — lineage + off-lineage panels by sub-cluster")
    dp.savefig(FIG / f"{TAGS[lin]}_unk_dotplot_markers.png", dpi=200)
    dp.show()

### §7d · The UNK annotation dict

Same rules as §6, one level down: every UNK sub-cluster ID must appear, every label must come from
`VOCAB[lin]`, and `UNK_CLUSTER_SIZES` locks the map to this exact sub-clustering. `UNK` stays the
right answer for a sub-cluster that is off-lineage (`RGS5`/`NOTCH3`, `KRT14`, `CD3D`, `TPSAB1`),
a doublet (two lineage cores co-expressed, high `doublet`), or low-quality (`med_genes` < 500,
`med_mito` > 15).

The cell rebuilds `subtype_fine` from `CLUSTER2FINE` before applying the refinement, so it is safe
to re-run and the two levels never stack. §8 and §9 then run unchanged on the refined labels.

In [ ]:
# ============================================================================
# §7d  UNK sub-cluster -> subtype_fine   (HAND-FILLED from §7c)
# ============================================================================
UNK2FINE = {
    "Myeloid": {"0": "UNK", "1": "UNK", "2": "cDC2", "3": "cDC2", "4": "Mac_infl", "5": "UNK", "6": "Mac_FOLR2", "7": "Mac_ISG", "8": "UNK", "9": "Mac_FOLR2", "10": "UNK", "11": "UNK", "12": "UNK"},
    "Fibroblast": {"0": "F_apCAF", "1": "UNK", "2": "UNK", "3": "F_inflammatory", "4": "F_reticular", "5": "UNK", "6": "UNK", "7": "F_inflammatory", "8": "UNK", "9": "F_papillary", "10": "UNK", "11": "UNK", "12": "F_apCAF", "13": "UNK", "14": "UNK", "15": "F_papillary", "16": "UNK", "17": "F_apCAF"},
}
UNK_CLUSTER_SIZES = {
    "Myeloid": {'0': 78, '1': 474, '2': 684, '3': 609, '4': 236, '5': 176, '6': 786, '7': 896, '8': 143, '9': 472, '10': 70, '11': 108, '12': 78},
    "Fibroblast": {'0': 1427, '1': 58, '2': 641, '3': 591, '4': 1095, '5': 718, '6': 704, '7': 194, '8': 444, '9': 570, '10': 527, '11': 1107, '12': 658, '13': 7, '14': 384, '15': 662, '16': 254, '17': 335},
}

UNK_MAPPED = True
for lin, u in U.items():
    key = UKEY[lin]
    cats = [str(c) for c in u.obs[key].cat.categories]
    now = {c: int((u.obs[key].astype(str) == c).sum()) for c in cats}
    m = UNK2FINE.get(lin, {})

    if (not m) or any(v == "" for v in m.values()):
        print(f"--- {lin}: UNFILLED. Paste into UNK2FINE[{lin!r}]: ---")
        print("    {" + ", ".join(f'"{c}": ""' for c in cats) + "}")
        print(f"--- and into UNK_CLUSTER_SIZES[{lin!r}]: ---\n    {now!r}\n")
        UNK_MAPPED = False
        continue

    assert set(m) == set(cats), (
        f"{lin} UNK: map keys != sub-clusters\n  missing: {natsorted(set(cats) - set(m))}\n"
        f"  extra  : {natsorted(set(m) - set(cats))}")
    bad = set(m.values()) - set(VOCAB[lin])
    assert not bad, f"{lin} UNK: labels outside the vocabulary: {sorted(bad)}"
    if UNK_CLUSTER_SIZES.get(lin):
        assert now == {str(k): int(v) for k, v in UNK_CLUSTER_SIZES[lin].items()}, (
            f"{lin} UNK: the sub-clustering changed since this map was authored.\n"
            f"  authored: {UNK_CLUSTER_SIZES[lin]}\n  now: {now}")
    else:
        print(f"!! {lin} UNK: UNK_CLUSTER_SIZES is empty — paste {now!r} to lock the map")

if not UNK_MAPPED:
    print("\nFill the dicts above from §7c, then re-run this cell — §8 uses the refined labels.")
else:
    for lin in LINEAGES:
        a = A[lin]
        # stage 1 rebuilt from scratch -> this cell is idempotent, the levels never stack
        a.obs["subtype_fine"] = pd.Categorical(a.obs[KEY[lin]].astype(str).map(CLUSTER2FINE[lin]))
        if lin not in U:
            continue
        u, key = U[lin], UKEY[lin]
        lab = a.obs["subtype_fine"].astype(str).to_numpy()
        new = pd.Series(u.obs[key].astype(str).map(UNK2FINE[lin]).to_numpy(),
                        index=u.obs["cell_id"].astype(str).to_numpy())
        idx = pd.Index(a.obs["cell_id"].astype(str)).get_indexer(new.index)
        assert (idx >= 0).all() and (lab[idx] == "UNK").all(), (
            f"{lin}: the UNK pool no longer maps onto UNK cells — re-run §7b")
        lab[idx] = new.to_numpy()
        a.obs["subtype_fine"] = pd.Categorical(lab)
        u.obs["subtype_fine"] = pd.Categorical(new.to_numpy())

        rescued = int((new != "UNK").sum())
        print(f"\n{lin}: {rescued:,}/{len(new):,} UNK cells rescued "
              f"({rescued / a.n_obs:.1%} of the lineage)")
        print(pd.crosstab(u.obs["parent_cluster"].astype(str), u.obs["subtype_fine"].astype(str))
              .loc[natsorted(u.obs["parent_cluster"].astype(str).unique())].to_string())
        print(f"{lin} subtype_fine after §7d:")
        print(a.obs["subtype_fine"].value_counts().to_string())

    for lin, u in U.items():
        a = A[lin]
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
        sc.pl.umap(u, color="subtype_fine", frameon=False, size=6, legend_loc="on data",
                   legend_fontsize=7, ax=axes[0], show=False,
                   title=f"{lin} UNK pool — refined ({u.obs['subtype_fine'].nunique()} labels)")
        sc.pl.umap(a, color="subtype_fine", frameon=False, size=3, legend_loc="on data",
                   legend_fontsize=7, ax=axes[1], show=False,
                   title=f"{lin} — subtype_fine after §7d "
                         f"({a.obs['subtype_fine'].nunique()} states)")
        for ax in axes:
            for coll in ax.collections:
                coll.set_rasterized(True)
        fig.savefig(FIG / f"{TAGS[lin]}_unk_umap_subtype_fine.png", dpi=200)
        plt.show(); plt.close(fig)

## §8 · The CCC collapse

A CCC level that clears `min_cells=25` in fewer than 5 CTCL donors cannot carry a claim, so it is
merged into its parent. The coverage table is computed **first**, and the collapse is written
against it.

In [ ]:
# ============================================================================
# §8  subtype_fine -> subtype_ccc, against per-donor coverage
# ============================================================================
def coverage(obs, col):
    ct = pd.crosstab(obs[col].astype(str), obs["donor"].astype(str))
    return pd.DataFrame({
        "n_cells": ct.sum(1),
        "n_donors_present": (ct > 0).sum(1),
        f"n_donors_ge{CCC_MIN_CELLS}": (ct >= CCC_MIN_CELLS).sum(1),
    }).assign(claimable=lambda d: d[f"n_donors_ge{CCC_MIN_CELLS}"] >= CCC_MIN_SAMPLES
              ).sort_values(f"n_donors_ge{CCC_MIN_CELLS}", ascending=False)


for lin in LINEAGES:
    ctcl = A[lin].obs[A[lin].obs["disease"].astype(str).isin(CTCL_DISEASES)]
    print(f"\n{lin} — CTCL window: {len(ctcl):,} cells, {ctcl['donor'].nunique()} donors")
    print(coverage(ctcl, "subtype_fine").to_string())

# "" drops the level from the CCC roster entirely.
FINE_TO_CCC = {
 "Myeloid": {
    "LC": "LC", "cDC1": "cDC", "cDC2": "cDC", "DC_LAMP3": "DC_LAMP3", "pDC": "pDC",
    "Mono": "Mono_moDC", "moDC": "Mono_moDC", "Mac_FOLR2": "Mac_FOLR2",
    "Mac_SPP1_TREM2": "Mac_SPP1_TREM2", "Mac_infl": "Mac_infl", "Mac_ISG": "Mac_infl",
    "Myeloid_prolif": "", "UNK": "",
 },
 "Fibroblast": {
    "F_papillary": "F_papillary", "F_reticular": "F_reticular",
    "F_mesenchymal": "F_mesenchymal", "F_myofibroblast": "F_mesenchymal",
    "F_inflammatory": "F_inflammatory", "F_apCAF": "F_apCAF", "F_prolif": "", "UNK": "",
 },
}

for lin in LINEAGES:
    a = A[lin]
    seen = set(a.obs["subtype_fine"].astype(str).unique())
    missing = seen - set(FINE_TO_CCC[lin])
    assert not missing, f"{lin}: fine labels with no collapse rule: {sorted(missing)}"
    a.obs["subtype_ccc"] = pd.Categorical(
        a.obs["subtype_fine"].astype(str).map(FINE_TO_CCC[lin]).replace("", np.nan))

    ctcl = a.obs[a.obs["disease"].astype(str).isin(CTCL_DISEASES) & a.obs["subtype_ccc"].notna()]
    cov = coverage(ctcl, "subtype_ccc")
    cov.to_csv(paths(TAGS[lin])["cov"])
    print(f"\n{lin} — CCC levels after the collapse:")
    print(cov.to_string())
    fail = cov.index[~cov["claimable"]].tolist()
    if fail:
        print(f"!! below the gate: {fail} — merge each into its parent above, or accept it as "
              f"report-only (nb39 excludes it from the claimable set either way).")
    else:
        print(f"all {len(cov)} levels clear {CCC_MIN_CELLS} cells in >= {CCC_MIN_SAMPLES} donors")

## §9 · Write the sidecar `25_delta_gene_axis`/39 read

In [ ]:
# ============================================================================
# §9  skin_myeloid_fibro_subtypes.csv + provenance
# ============================================================================
# subtype_ccc stays NULL (not the string "nan") for the levels §8 dropped — that null is the
# signal build_ccc_celltype_sub() uses to drop the cell.
merged = pd.concat([
    pd.DataFrame({
        "cell_id": A[lin].obs["cell_id"].astype(str).to_numpy(),
        "lineage": lin,
        "leiden_sub": A[lin].obs[KEY[lin]].astype(str).to_numpy(),
        "subtype_fine": A[lin].obs["subtype_fine"].astype(str).to_numpy(),
        "subtype_ccc": A[lin].obs["subtype_ccc"].astype(object)
                       .where(A[lin].obs["subtype_ccc"].notna()).to_numpy(),
    }) for lin in LINEAGES], ignore_index=True)

assert merged["cell_id"].is_unique, "duplicate cell_id across the two lineages"
assert len(merged) == N_MYE + N_FIB, (len(merged), N_MYE + N_FIB)
merged.to_csv(SUBTYPE_CSV, index=False)
print("wrote", SUBTYPE_CSV, merged.shape,
      f"| {int(merged['subtype_ccc'].isna().sum()):,} cells dropped from the CCC roster")
print(pd.crosstab(merged["lineage"], merged["subtype_ccc"].fillna("(dropped)")).to_string())

PROV_JSON.write_text(json.dumps({
    "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "10c_skin_myeloid_fibro_reannotation.ipynb",
    "source_h5ad": str(SRC_H5AD), "subset_h5ad": str(SUBSET_H5AD), "label_source": str(LABEL_CSV),
    "seed": SEED, "n_hvg": N_HVG, "n_latent": N_LATENT,
    "hvg_min_cells_per_batch": HVG_MIN_CELLS_PER_BATCH,
    "scvi_batch_key": "study", "scvi_categorical_covariate_keys": ["donor"],
    "scvi_models": {lin: str(paths(TAGS[lin])["scvi"]) for lin in LINEAGES},
    "leiden_res": LEIDEN_RES,
    "unk_leiden_res": globals().get("UNK_RES"),
    "ccc_min_cells": CCC_MIN_CELLS, "ccc_min_samples": CCC_MIN_SAMPLES,
    "ctcl_diseases": CTCL_DISEASES,
    "cluster2fine": CLUSTER2FINE, "cluster_sizes": CLUSTER_SIZES,
    "unk2fine": globals().get("UNK2FINE"), "unk_cluster_sizes": globals().get("UNK_CLUSTER_SIZES"),
    "fine_to_ccc": FINE_TO_CCC,
    "n_cells": N_CELLS,
    "ccc_levels": sorted(set(merged["subtype_ccc"].dropna())),
}, indent=2, default=str))
print("wrote", PROV_JSON)

print("\nPaste into ccc_data_sub.py:")
for lin, name in (("Myeloid", "MYELOID_LEVELS"), ("Fibroblast", "FIBRO_LEVELS  ")):
    lv = sorted(set(merged.loc[merged.lineage == lin, "subtype_ccc"].dropna()))
    print(f"  {name} = {lv}")

### Next

1. `ccc_data_sub.py` — paste `MYELOID_LEVELS` / `FIBRO_LEVELS` from §9.
2. `old/ccc_v1_04_subtype_descriptive.ipynb` — §0 joins this sidecar onto `ccc_skin.h5ad` by `cell_id`
   (the object already holds every myeloid/fibroblast cell, so it is not rebuilt).
3. `old/ccc_v1_05_subtype_headline_figures.ipynb`.

`ccc_skin_pseudobulk_full.parquet` is keyed by the **old** pooled grouping and is stale for these
levels. It feeds only the deferred donor-level differential phase — re-run `jobs/run_ccc_build.py`
if that starts.